# Prosjekt 2

Info

In [1]:
# Necessary libraries
import scipy
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import jax.lax as lax
import time
import numpy as np
from jax.experimental.ode import odeint
from functools import partial

jax.config.update("jax_enable_x64", True)

## Model Problems

In [2]:
def r(x: jax.Array) -> float:
    return jnp.sqrt(x[0]**2+ x[1]**2)

def r2(x: jax.Array) -> float:
    return x[0]**2+ x[1]**2

#### Charged Particle

In [ ]:
def B_cp(x: jax.Array) -> jax.Array:
    return jnp.array([0, 0, r(x)])


def A_cp(x: jax.Array) -> jax.Array:
    return 1/3 * jnp.array([-x[1]*r(x), x[0]*r(x), 0])

def dA_cp(x: jnp.array) -> jnp.array:

    r = jnp.sqrt(x[0]**2 + x[1]**2)

    return jnp.array([[-1/3*x[0]*x[1]/ r, -1/3 * (r + x[1]**2/ r), 0*r],
                        [1/3 * (r + x[0]**2/r), 1/3 * x[0] * x[1]/r, 0*r],
                        [0*r,                   0*r,                      0*r]])

def U_cp(x: jax.Array) -> float:
    rr = jnp.clip(r(x), 1e-20, None) # To make sure the I dont get divide by zero errors
    return 1 / (100 * rr)

def gradU_cp(x: jax.Array) -> jax.Array:
    rr= jnp.clip(r(x), 1e-20, None) # TO make sure the I dont get divide by zero errors
    return -1/ 100 *jnp.array([x[0]/rr**3, x[1]/rr**3, 0])

# System of differential equations
@jax.jit
def system_cp(t: jax.Array, Y: jax.Array) -> jax.Array:
    x = Y[:3]
    v = Y[3:]
    dxdt = v
    dvdt = jnp.cross(v, B_cp(x)) - gradU_cp(x)
    return jnp.concatenate((dxdt, dvdt))

charged_particle = dict(
    name = "charged particle",
    x0 = jnp.array([0, 1, 0.1]),
    v0 = jnp.array([0.09, 0.05, 0.20]),
    B = B_cp,
    A = A_cp,
    dA = dA_cp,
    U = U_cp,
    gradU = gradU_cp,
    system = system_cp
)

#### Tokamak

In [ ]:
def B_tok(x: jax.Array) -> jax.Array:
    """
    Returns the forces of the magnetic Field
    """

    rr2 = jnp.maximum(r2(x), 1e-20) # TO make sure the I dont get divide by zero errors
    rr = jnp.maximum(r(x), 1e-20) # TO make sure the I dont get divide by zero errors

    return jnp.array([-(2 * x[1] + x[0]*x[2]) / (2 *rr2),
                          (2*x[0] - x[1]*x[2]) / (2* rr2),
                                    (r(x) -1) / (2* rr)])


def A_tok(x: jax.Array) -> jax.Array:

    rr2 = jnp.maximum(r2(x), 1e-20) # TO make sure the I dont get divide by zero errors
    rr = jnp.maximum(r(x), 1e-20) # TO make sure the I dont get divide by zero errors

    return jnp.array([0, 
                     1/2  * ( x[0] - jnp.log(rr + x[0]) ), 
                     -1/2 * (x[2] * jnp.arctan(x[1]/x[0]) + jnp.log(rr2)) ])

def dA_tok(x: jnp.array) -> jnp.array:

    """
    Analytically computed Jacobian for A(x) in the Tokamak magnetic field

    x: position on form (x1, x2, x3)

    Returns a 3x3 Jacobian matrix
    """

    r = jnp.sqrt(x[0]**2+ x[1]**2)
    r2 = x[0]**2+ x[1]**2

    return jnp.array([ [0*r,     0*r     , 0*r],
                      [1/2 - (x[0]/r + 1) / (2 * (x[0] + r)), -x[1] / (2*(x[0] + r) * r), 0*r],
                       [-x[0]/r2 + x[1]*x[2]/(2 * r2), -x[1]/ r2 - x[2] / (2 * x[0] * (1 + (x[1]/ x[0])**2)), -jnp.atan2(x[1], x[0]) / 2]])


def U_tok(x: jax.Array) -> float:
    return 0.0

def gradU_tok(x: jax.Array) -> jax.Array:
    return jnp.array([0.0, 0.0, 0.0])


# # System of differential equations
@jax.jit
def system_tok(t: float, Y: jax.Array) -> jax.Array:
    x = Y[:3]
    v = Y[3:]
    dxdt = v
    dvdt = jnp.cross(v, B_tok(x)) - gradU_tok(x)
    return jnp.concatenate((dxdt, dvdt))

tokamak = dict(
    name = "tokamak",
    x0 = jnp.array([1.2, 0.0, 0.0]),
    v0 = jnp.array([0.0, 4.816e-4, -2.059e-3]),
    B = B_tok,
    A = A_tok,
    dA = dA_tok,
    U = U_tok,
    gradU = gradU_tok,
    system = system_tok
)

## Helper functions

In [5]:
def energy(problem, x, v):
    U = jax.vmap(problem["U"], in_axes=1)
    return 0.5 * jnp.sum(v**2, axis=0) + U(x)


def reference_solution(problem, t_eval):
    Y0 = jnp.concatenate([problem["x0"], problem["v0"]])

    #jax odeint expects system to have arg (Y, t) instead of (t, Y)
    def f(Y, t):
        return problem["system"](t, Y)

    Y = odeint(f, Y0, jnp.asarray(t_eval), rtol=1e-12, atol=1e-12).T

    return t_eval, Y[:3], Y[3:]

def timed(solve, problem, h, T):

    #lets jit warm up one round before measuring the actual compute time
    jax.block_until_ready(solve(problem, h, T))
    
    start = time.perf_counter()
    out = jax.block_until_ready(solve(problem, h, T))

    return out, time.perf_counter() - start

## Solvers

### Boris

In [ ]:
@partial(jax.jit, static_argnames=("B", "gradU", "T", "h"))
def Boris_method(v_init: jnp.array, x0: jnp.array, B, gradU, T: int, h: float) -> tuple[jnp.array, jnp.array]:

    """
    Boris_method, calculates the position and velocity of the particle at time t.

    x0: Iinitial position, array of size 3.
    v0: Iinitial velocity, array of size 3.
    T:  Total time.
    h:  Steplength in time.

    return: arrays of positions and velocities.
    """

    N = int(round(T/h))

    x = x0
    v = v_init

    def Boris_step(x: jnp.array, v: jnp.array) -> tuple[jnp.array, jnp.array]:
        """
        Calculates one step of the Boris method


        x: Current position, array of size 3.
        v: Current velocity, array of size 3.
        h:  Steplength in time.
        
        return: x_next, v_nect
        """

        v_minus = v -  h/2 * gradU(x)    #v_minus

        t = h/2 * B(x)
        s = 2 * t / (1 + t @ t)
        v_prime = v_minus + jnp.cross(v_minus, t)
        v_plus = v_minus + jnp.cross(v_prime, s) 

        v_next = v_plus - h/2 * gradU(x)
        x_next = x + h * v_next

        return x_next, v_next

    def body(carry, _):
        x, v  = carry
        x, v = Boris_step(x, v)

        return (x, v), (x,v)

    (_, _), trajectory = lax.scan(body, (x, v), xs = None, length = N+1)

    positions, velocities = trajectory

    positions = jnp.concatenate([x0[None, :], positions], axis=0)

    velocities = jnp.concatenate([v_init[None, :], velocities], axis=0)
    v_averages = 1/2 * (velocities[:-1] + velocities[1:])
    return positions[:-1].T, v_averages.T

#wrapper function, "main"
def boris(problem, h, T):
    
    # v(-h/2): odeint needs increasing t, so we integrate -f forwards instead of f backwards
    Y0 = jnp.concatenate([problem["x0"], problem["v0"]])
    Y_half = odeint(lambda Y, t: -problem["system"](t, Y), Y0, jnp.array([0.0, h/2]),
                    rtol=1e-12, atol=1e-12)
    v_init = Y_half[-1, 3:]

    x, v = Boris_method(v_init, problem["x0"], problem["B"], problem["gradU"], T, h)
    N = int(round(T/h))
    t = h * jnp.arange(N + 1)
    return t, x, v

### RK4

In [ ]:
def RK4_step(t:jax.Array, Yn:jax.Array, system:jax.Array, h:float) ->jax.Array:
    """
    t: time (is ignored here)
    Yn: [x1,x2,x3,v1,v2,v3]
    system: gives dY/dt (RHS of differential equation)
    h: steplength
    returns: Yn_new

    """
    k1 = h * system(t, Yn)
    k2 = h * system(t, Yn + 0.5*k1)
    k3 = h * system(t, Yn + 0.5*k2)
    k4 = h * system(t, Yn + k3)

    Yn_new = Yn + 1/6*(k1 + 2*k2 + 2*k3 + k4)

    return Yn_new

@partial(jax.jit, static_argnames=("system", "N"))
def rk4_subwrap(N: int,  Y0:jax.Array, system, h:float) -> jax.Array:
    """
    N: number of steps
    Y0: initial Y vector
    system: RHS differential eqt
    h: step length
    """
    def body(Yn, _):
        Yn_new = RK4_step(0.0, Yn, system, h)
        return Yn_new, Yn_new

    Y_final, trajectory = lax.scan(body, Y0, xs=None, length=N)
    
    Y = jnp.concatenate((Y0[None, :], trajectory), axis=0).T

    return Y

#wrapper function, "main"
def rk4(problem, h, T):
    N = int(round(T / h))
    Y0 = jnp.concatenate([problem["x0"], problem["v0"]])
    Y = rk4_subwrap(N, Y0, problem["system"], h)
    t = h * jnp.arange(N+1)
    return t, Y[:3], Y[3:]

### Multistep

In [ ]:
# Constants for fourth order symmetric multistep
a1 = -0.7
a2 = 0.1
a3 = 0.9

beta0 = (1/3) * (20*a1*a2*a3 - 4*(a1*a2 + a1*a3 + a2*a3) - 28*(a1 + a2 + a3) - 52)
beta1 = (1/3) * (2*a1*a2*a3 + 14*(a1*a2 + a1*a3 + a2*a3) + 26*(a1 + a2 + a3) + 38)

alpha_list = jnp.array([
    1,           # α₋₄
    -7/5,        # α₋₃
    9/25,        # α₋₂
    22/125,      # α₋₁
    -34/125,     # α₀
    22/125,      # α₁
    9/25,        # α₂
    -7/5,        # α₃
    1            # α₄
])

@jax.jit
def delta_yj(y: jnp.array) -> float:
    """
    Five point central difference centered in y[2]

    y: array on form (y_j-2, y_j-1, y_j, y_j+1, y_j+2). Each y has 3 components

    Returns the central difference
    """

    return 1/12 * (y[:,0] - 8 * y[:,1] + 8* y[:,3] - y[:,4]) 

@jax.jit
def F_j_jax(A_x :jnp.array, x :jnp.array,  h: float, U_grad_xj: jnp.array, A_jacobian: jnp.array) -> jnp.array:
    """
    Fourth order symmetric multistep method help function

    x: array with shape (x_j-2, x_j-1, x_j, x_j+1, x_j+2). Each x has 3 compontents
    h: steplength
    U_grad: potential gradient function

    Returns a 3 component array
    """

    x_j = x[:, 2]
    delta_Aj = delta_yj(A_x)/ h
    vj = delta_yj(x)/ h

    return  A_jacobian.T @ vj - delta_Aj - U_grad_xj


def fourth_order_sym_multistep_step_fast(x: jnp.array, alpha: jnp.array, h: float, beta0: float, beta1: float, F: tuple) -> jnp.array:

    """
    Takes a step in the fourth order symmetrical multistep method

    x: array on form (x_n-4, x_n-3, ... , x_n+4)
    alpha: array on form (alpha_n-4, alpha_n-3, ... , alpha_n+4)
    h: steplength
    beta0, beta1: method parameters, 
    F: tuple on form (F_n-1, F_n, F_n+1)

    Returns an array x with updated x_n+4
    """

    prev_steps = jnp.sum(x * alpha[:-1], axis = 1)
    x_n4 = 1/ alpha[-1] *( h**2 * (beta1 * F[0] + beta0 * F[1] + beta1 * F[2]) - prev_steps)
    #x[-1] = x_n4

    return x_n4


@partial(jax.jit, static_argnames=("t_size", "U_grad", "A", "jacobian"))
def fourth_order_sym_multistep_lax(x: jnp.array, v: jnp.array, t_size: int, alpha: jnp.array, h: float, beta0: float, beta1: float, U_grad: callable, A: callable, jacobian: callable) -> jnp.array:

    """
    Fourth order symmetrical multistep method

    x: array with same shape as the final position array, except with the inital 8 values computed
    v: same as x but for velocity
    t_size: number of time points
    alpha: array on form (alpha_n-4, alpha_n-3, ... , alpha_n+4)
    h: steplength
    beta0, beta1: method parameters, 
    U_grad: potential gradient function
    A: magnetic field vector potential function
    Jacobian: Analytically computed jacobian of A 

    Returns a dict with x and v as keys
    """
    A_vec = jax.vmap(A, in_axes=1, out_axes=1)

    def scan_step(carry: tuple, _) -> jnp.array:

        """
        This function does the iterations in the multistep method. We initially implemented the 
        multistep method with numpy and a for loop here, but this took to much time. This function combined with lax.scan
        computes the values at every iteration simultanously

        carry: tuple of values which are updated for each iteration.
               Has the form (x_window, F_prev, F_n, F_next) 
               x_window: array of x_n-4 to x_n+3 (x_n+4 is the value we compute in each iteration)
               F_prev, F_n, F_next: F centered at x_n-1, x_n, x_n+1 respectively


        Returns new carry which are the updated iterative values, and x_last which is 
        an array of every x computed up to this point
        """

        x_window, v_current, F_prev, F_n, F_next = carry 

        # Compute new position
        x_last = fourth_order_sym_multistep_step_fast(x_window, alpha, h, beta0, beta1, (F_prev, F_n, F_next))

        # New window of x_n-4 to x_n+3 (x_n+4 is the value we compute in each iteration)
        new_x_window = jnp.concatenate([x_window[:, 1:], x_last[:, None]],axis=1)

        # Compute velocity as five point difference
        v_current =  delta_yj(new_x_window[:, -5:]) / h 

        # Update the F functions in each itearation
        F_prev = F_n                                        # Need five x-values centered around -5
        F_n = F_next                                        # Need five x-values centered around -4
        A_jacobi_next = jacobian(new_x_window[:, -3])
        U_grad_next = U_grad(new_x_window[:, -3])
        A_next = A(new_x_window[:, -5:])
        F_next = F_j_jax(A_next, new_x_window[:, -5:], h, U_grad_next, A_jacobi_next)     # Need five x-values centered around -3

        # Save current values for next iteration
        new_carry = (new_x_window, v_current, F_prev, F_n, F_next)
        
        return new_carry, (x_last, v_current)

    # Find initial values of the Fs
    A_jacobi_next = jacobian(x[:, 3])
    U_grad_next = U_grad(x[:, 3])
    A_next = A(x[:,  (1): (6)])
    F_prev = F_j_jax(A_next, x[:, (1): (6)],  h, U_grad_next, A_jacobi_next)     # Need five x-values centered around 3

    A_jacobi_next = jacobian(x[:, 4])
    U_grad_next = U_grad(x[:, 4])
    A_next = A(x[:,  (2): (7)])
    F_n = F_j_jax(A_next, x[:, (2): (7)], h, U_grad_next, A_jacobi_next)         # Need five x-values centered around 4

    A_jacobi_next = jacobian(x[:, 5])
    U_grad_next = U_grad(x[:, 5])
    A_next = A(x[:, (3): (8)])
    F_next = F_j_jax(A_next, x[:, (3): (8)], h, U_grad_next, A_jacobi_next)      # Need five x-values centered around 5

    
    # Inital values for lax.scan
    init_x_window = x[:, :8]
    initial_carry = (init_x_window, v[:, 7], F_prev, F_n, F_next)

    # lax.scan computes every value in the multistep iteration simultanously
    final_carry, final_xv = lax.scan(scan_step, initial_carry,  xs= None, length = t_size - 9)

    # Final x holds every x-value computed in the lax scan
    # x holds the 8 initial values
    x_full = jnp.concatenate([x[:, :8], final_xv[0].T],axis=1)
    v_full = jnp.concatenate([v[:, :8], final_xv[1].T],axis=1)

    return {"x": x_full, "v": v_full}

def create_grids(x_0: jnp.array, v_0: jnp.array,  h: float,  t_0: float, t_f: float, B: callable, U_grad: callable):
      
    # Create grids
    t = jnp.arange(t_0, t_f, h)
    x = jnp.zeros(( 3, t.size))
    v = jnp.zeros(( 3, t.size))
    
    # Find first eight x values with solve_ivp RK45
    y_0 = jnp.concatenate((x_0, v_0), axis = None)
    init_sol = scipy.integrate.solve_ivp(charged_particle_eq, (t_0, t[7]), y_0, args = (B, U_grad), dense_output=True, method = "RK45", rtol=1e-10, atol=1e-12)
    x = x.at[:,:8].set(jnp.asarray(init_sol.sol(t[0:8])[:3]))
    v = v.at[:,:8].set( jnp.asarray(init_sol.sol(t[0:8])[3:]))

    return x, v, t